# Bounded Self-Instruct → Qwen3-8B SFT → Lighteval loop

GPT-6 Luna is the parent model, accessed through the OpenAI Responses API. The student is loaded once below and updated with QLoRA. Each round generates new scenarios, records/cleans the student's predictions, obtains validated parent targets, runs SFTTrainer, then evaluates on a frozen synthetic development set using native Lighteval custom metrics.

Student predictions are diagnostics. The student's forward pass in SFTTrainer is trained against parent completion tokens; incorrect sampled student responses are never used as labels. This notebook does not claim independent medical validation. Run cells in order on a CUDA GPU with Internet enabled.


In [ ]:
from pathlib import Path
import os
import sys

# Attach/clone the complete repository, not just this notebook.
# Override PROJECT_ROOT explicitly if your checkout is elsewhere.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "requirements-training.txt").is_file():
    candidates = []
    for parent in (Path("/kaggle/working"), Path("/kaggle/input")):
        if parent.exists():
            candidates.extend(parent.rglob("requirements-training.txt"))
    candidates = [p.parent for p in candidates if (p.parent / "run_self_instruct.py").is_file()]
    if len(candidates) != 1:
        raise RuntimeError("Set PROJECT_ROOT to the complete repository checkout, then rerun this cell.")
    PROJECT_ROOT = candidates[0]
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
REQUIREMENTS = str(PROJECT_ROOT / "requirements-training.txt")
print("Project:", PROJECT_ROOT)


In [ ]:
%pip install -q -r "{REQUIREMENTS}"
# If an earlier cell imported incompatible library versions, restart the kernel
# once after installation, then rerun from the top.


In [ ]:
import os

# Only the OpenAI parent needs a credential. Qwen3-8B loads anonymously below.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY").strip()
    except Exception as exc:
        raise RuntimeError("Add OPENAI_API_KEY to Kaggle Secrets and enable notebook access.") from None
if not os.environ.get("OPENAI_API_KEY", "").strip():
    raise RuntimeError("OPENAI_API_KEY is empty.")
print("OpenAI credential loaded. Hugging Face login is not required.")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from distillation_loop.evaluation import require_training_versions

require_training_versions()
MODEL_NAME = "Qwen/Qwen3-8B"
assert torch.cuda.is_available(), "Enable a CUDA GPU in Kaggle settings."

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# One-process QLoRA on GPU 0; no second model copy is loaded for Lighteval.
student = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=False,  # Ignore stale environment/cached tokens for this public model.
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)
student.eval()
print("Student loaded:", MODEL_NAME)
print("Memory footprint (GiB):", round(student.get_memory_footprint() / 1024**3, 2))


## Loop settings

Defaults: at most **5 rounds**, **72 new training scenarios per round**, and **3 frozen development examples per task** (54 examples). Development generation and reference answers use paid parent API calls on first run; subsequent runs reuse the frozen set. Every round has an additional optimizer-step cap.

The scores below are conservative reference-based custom metrics, not a medical benchmark: label accuracy for classification, field/value F1 for extraction, code-set F1 for coding, and source-gated ROUGE-L for free-text tasks. All 18 tasks must have development coverage. You can edit thresholds and limits before the first run; changing them after a run starts requires a fresh output directory.


In [ ]:
from self_instruct_pipeline.config import load_config
from self_instruct_pipeline.persistence import atomic_json
from distillation_loop.core import load_loop_config
from distillation_loop.student import StudentRuntime

parent_config = load_config(PROJECT_ROOT / "config/self_instruct.json")
loop_config = load_loop_config(str(PROJECT_ROOT / "config/distillation_loop.json"))

# Editable before starting a new run:
loop_config["max_rounds"] = 5
loop_config["thresholds"] = {
    "macro_task_score": 0.90,
    "minimum_task_score": 0.80,
    "schema_evidence_valid_rate": 0.98,
}
RUN_DIR = (Path("/kaggle/working/distillation_loop_openai") if Path("/kaggle/working").exists()
           else PROJECT_ROOT / "outputs/distillation_loop_openai")
RUN_DIR.mkdir(parents=True, exist_ok=True)
loop_config["output_dir"] = str(RUN_DIR)
# Validate notebook overrides using the same configuration loader.
atomic_json(RUN_DIR / "requested_loop_config.json", loop_config)
loop_config = load_loop_config(str(RUN_DIR / "requested_loop_config.json"))
student_runtime = StudentRuntime(student, tokenizer, loop_config)
student_runtime.prepare()  # Freezes the 4-bit base and enables trainable LoRA adapters.
print("Max rounds:", loop_config["max_rounds"])
print("Thresholds:", loop_config["thresholds"])


In [ ]:
# Optional request preview: this cell makes no parent API calls.
import copy
from self_instruct_pipeline.pipeline import Pipeline
preview_config = copy.deepcopy(parent_config)
preview_config["paths"]["output_dir"] = str(RUN_DIR / "preview")
preview = Pipeline(preview_config).plan()
print("Task quotas:", preview["task_quotas"])
print("Seed demonstrations:", preview["seed_ids"])


## Run or resume

This cell calls the parent API and performs GPU training. Rerun it with the same configuration/output directory to resume. It stops on satisfactory development scores, the maximum round count, or an explicit API/validation/training error. It never silently skips failed evaluation or loops forever. Each completed round saves its adapter, training metrics, student diagnostics, and Lighteval scores. The best adapter is restored at the end; a fully completed rerun does not train again. An unfinished round restarts from the last completed checkpoint.


In [ ]:
from distillation_loop.controller import run_loop

result = run_loop(student_runtime, parent_config, loop_config)
student = student_runtime.model  # Best saved adapter is restored at the end.
print("Stop reason:", result["status"])
print("Completed rounds:", result["completed_rounds"])
print("Best checkpoint:", result["best_checkpoint"])
print("Best Lighteval macro score:", result["best_report"]["macro_task_score"])


In [ ]:
# Inspect history without rerunning inference or training.
for entry in result["history"]:
    scores = entry["scores"]
    print({"round": entry["round"], "macro": scores["macro_task_score"],
           "minimum_task": scores["minimum_task_score"],
           "schema_evidence_valid": scores["schema_evidence_valid_rate"]})
